<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/count_price_related_posts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os
from datetime import datetime

# --- Configuration ---

# Input File Configuration
INPUT_DIR = '/content/drive/MyDrive/world-inflation/data/reddit/production/'
INPUT_FILENAME = 'Frugal_submissions_2012_2022.tsv'
INPUT_PATH = os.path.join(INPUT_DIR, INPUT_FILENAME)

# Output Google Sheet Configuration
OUTPUT_SPREADSHEET_NAME = 'monthly_count_results'
OUTPUT_WORKSHEET_NAME = 'subfrugal'

# Date Filtering Configuration
START_DATE = '2012-01-01'
END_DATE = '2022-12-31'

# Output Header Configuration
HEADER_COLUMNS = ['month', 'price-related_posts']

# Check if input file exists before proceeding
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"The file was not found at: {INPUT_PATH}")
else:
    print(f"Configuration set. Target file found: {INPUT_PATH}")

In [ ]:
# Load the TSV file
# on_bad_lines='skip' is used to prevent crashes if the TSV is malformed
df = pd.read_csv(INPUT_PATH, sep='\t', on_bad_lines='skip')

# Convert created_date to datetime objects
df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')

# Drop rows where date conversion failed
df = df.dropna(subset=['created_date'])

# Filter data based on the configured date range
mask = (df['created_date'] >= START_DATE) & (df['created_date'] <= END_DATE)
df_filtered = df.loc[mask].copy()

# Set the date as index for resampling
df_filtered.set_index('created_date', inplace=True)

# Resample by Month Start ('MS') and count the number of rows
# We use .size() to get a simple count of rows per period
monthly_counts = df_filtered.resample('MS').size()

# Create a clean DataFrame for export
result_df = monthly_counts.reset_index()
result_df.columns = HEADER_COLUMNS

# Format the 'month' column to 'YYYY-MM' string format as per requirement
result_df['month'] = result_df['month'].dt.strftime('%Y-%m')

# Preview the results
print(f"Aggregation complete. Total months: {len(result_df)}")
print(result_df.head())

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

# Authenticate the user
auth.authenticate_user()

# Get credentials and authorize the client
creds, _ = default()
gc = gspread.authorize(creds)

print("Authentication successful.")

In [ ]:
try:
    # Open the Spreadsheet by name
    sh = gc.open(OUTPUT_SPREADSHEET_NAME)

    # Try to retrieve the worksheet, create if it does not exist
    try:
        worksheet = sh.worksheet(OUTPUT_WORKSHEET_NAME)
        print(f"Worksheet '{OUTPUT_WORKSHEET_NAME}' found. Clearing existing content.")
        worksheet.clear()
    except gspread.exceptions.WorksheetNotFound:
        print(f"Worksheet '{OUTPUT_WORKSHEET_NAME}' not found. Creating new worksheet.")
        worksheet = sh.add_worksheet(title=OUTPUT_WORKSHEET_NAME, rows=1000, cols=20)

    # Prepare data for upload (convert DataFrame to list of lists)
    # Include headers
    data_to_export = [result_df.columns.values.tolist()] + result_df.values.tolist()

    # Update the worksheet
    worksheet.update(data_to_export)

    print(f"Successfully wrote {len(result_df)} rows to '{OUTPUT_SPREADSHEET_NAME}' > '{OUTPUT_WORKSHEET_NAME}'.")

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: The Google Spreadsheet named '{OUTPUT_SPREADSHEET_NAME}' was not found.")
    print("Please create the blank Spreadsheet in your Drive first, or ensure the name is exact.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")